# Análise de Localização — Unitree Go2 + BotBrain Pro

**Disciplina:** Ciência de Dados — FEI Mestrado  
**Descrição:** Análise da qualidade de localização do RTABMAP utilizando uma ou duas câmeras no robô Go2.

---

### Fontes de Dados
| Arquivo | Descrição |
|---------|-----------|
| `localization_log.csv` | Métricas por atualização dos tópicos `/rtabmap/info` e `/localization_pose` (inliers, covariância, pose, etc.) |
| `plan_log.csv` | Poses do caminho planejado pelo tópico `/plan`, agrupadas por `plan_id` |

### Seções
1. **Carregamento e Processamento dos Dados** — leitura dos arquivos CSV e pré-processamento
2. **Visualização dos Caminhos** — caminho planejado vs trajetória real do robô
3. **Qualidade da Localização** — inliers, razão de hipótese e covariância ao longo do tempo

### 1. Carregamento e Processamento dos Dados

Os dados foram coletados em 20 execuções do robô Go2, divididas em dois grupos:
- **Runs 1–10:** robô equipado com **duas câmeras**
- **Runs 11–20:** robô equipado com **uma câmera**

Cada run possui dois arquivos de log: um com as métricas de localização e outro com o caminho planejado pelo stack de navegação NAV2.

In [38]:
import math
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.interpolate import interp1d
from scipy.stats.mstats import winsorize
from plotly.subplots import make_subplots

DEBUG = False  # Defina como True para imprimir os prints de debug
SAVE_GRAPHS = True  # Defina como True para exportar todos os gráficos como SVG em graphs/


logs_dir = Path("localization_analysis/data/logs")

loc_csv_paths = sorted(logs_dir.glob("logger_csv_*/localization_log.csv"), key=lambda p: int(p.parent.name.split('_')[-1]))
runs_loc_df    = [pd.read_csv(loc_path) for loc_path in loc_csv_paths] 

plan_csv_paths = sorted(logs_dir.glob("logger_csv_*/plan_log.csv"), key=lambda p: int(p.parent.name.split('_')[-1]))
runs_plan_df    = [pd.read_csv(plan_path) for plan_path in plan_csv_paths] 

In [39]:
# Função para plot formatado para artigos de IEEE 

_IEEE_FONT       = 'Times New Roman'
_IEEE_AXIS_SIZE  = 16
_IEEE_TITLE_SIZE = 18

def apply_ieee_style(fig):
    _axis = dict(
        showline=True, linecolor='black', linewidth=1,
        mirror=True,
        showgrid=True, gridcolor='lightgrey', gridwidth=0.5,
        ticks='outside', ticklen=4, tickwidth=1, tickcolor='black',
        tickfont=dict(family=_IEEE_FONT, size=_IEEE_AXIS_SIZE, color='black'),
        title_font=dict(family=_IEEE_FONT, size=_IEEE_AXIS_SIZE, color='black'),
    )
    fig.update_xaxes(**_axis)
    fig.update_yaxes(**_axis)
    fig.update_annotations(font=dict(family=_IEEE_FONT, size=_IEEE_TITLE_SIZE, color='black'))
    fig.update_layout(
        paper_bgcolor='white',
        plot_bgcolor='white',
        title=dict(
            font=dict(family=_IEEE_FONT, size=_IEEE_TITLE_SIZE + 2, color='black'),
            x=0.5, xanchor='center',
        ),
        legend=dict(font=dict(family=_IEEE_FONT, size=_IEEE_AXIS_SIZE)),
        margin=dict(l=60, r=20, t=60, b=60),
    )
    return fig

from pathlib import Path
GRAPHS_DIR = Path('graphs')
GRAPHS_DIR.mkdir(exist_ok=True)

def save_fig(fig, name):
    if SAVE_GRAPHS:
        fig.write_image(GRAPHS_DIR / name)


In [40]:
print(f' Localization Dataframes: {len(runs_loc_df)}\n Plan Dataframes {len(runs_plan_df)}') # Número de DataFrames carregados

 Localization Dataframes: 20
 Plan Dataframes 20


In [41]:
# Pega o primeiro caminho calculado pela pilha NAV2 (plan_id = 1)
plans_df = [df[df['plan_id'] == df['plan_id'].min()] for df in runs_plan_df]
plans_df[0]

,plan_id,pose_index,x,y
0,1,0,-1.9519,1.0049
1,1,1,-1.9169,0.9549
2,1,2,-1.8827,0.9049
3,1,3,-1.8499,0.8549
4,1,4,-1.8184,0.8049
...,...,...,...,...
125,1,125,1.7117,1.5549
126,1,126,1.7184,1.6049
127,1,127,1.7269,1.6549
128,1,128,1.7370,1.7049


In [42]:
# Modificando o nome para paths_df
paths_df = runs_loc_df
paths_df[0]

,timestamp_sec,camera_mode,node_id,inliers,matches,inlier_ratio,hypothesis_ratio,loop_closure_id,wm_size,detection_time_ms,total_time_ms,pos_x,pos_y,yaw,cov_xx,cov_yy,cov_yaw,cov_pos_trace
0,1.776904e+09,double,19302,0,0,0.0000,0.0,0,977,378.34,389.37,-1.9692,1.0266,2.3477,0.016145,0.017012,0.014193,0.033157
1,1.776904e+09,double,19303,0,0,0.0000,0.0,0,977,314.39,330.24,-1.9690,1.0271,2.3480,0.016145,0.017012,0.014193,0.033157
2,1.776904e+09,double,19304,0,0,0.0000,0.0,0,977,478.89,492.52,-1.9684,1.0275,2.3484,0.016145,0.017012,0.014193,0.033157
3,1.776904e+09,double,19305,0,0,0.0000,0.0,0,977,277.65,293.06,-1.9682,1.0279,2.3486,0.016145,0.017012,0.014193,0.033157
4,1.776904e+09,double,19306,3,33,0.0066,1.0,0,977,264.31,279.39,-1.8453,0.8837,2.6791,0.017145,0.018012,0.015193,0.035157
5,1.776904e+09,double,19307,0,0,0.0000,0.0,0,977,341.45,347.20,-1.6645,0.7281,-2.2869,0.018145,0.019012,0.016193,0.037157
6,1.776904e+09,double,19308,26,160,0.0582,1.0,0,977,284.78,307.10,-1.6312,0.3937,-1.3213,0.019145,0.020012,0.017193,0.039157
7,1.776904e+09,double,19309,0,0,0.0000,0.0,0,977,398.18,405.12,-1.4301,-0.0850,-1.3658,0.020145,0.021012,0.018193,0.041157
8,1.776904e+09,double,19310,0,0,0.0000,0.0,0,977,321.11,328.92,-1.3238,-0.5538,-1.2732,0.021145,0.022012,0.019193,0.043157
9,1.776904e+09,double,19311,34,157,0.0787,0.0,0,977,331.10,342.50,-1.3135,-0.9109,-0.9104,0.002459,0.002524,0.002298,0.004983


### 2. Visualização dos Caminhos

Comparação entre o caminho planejado pelo NAV2 e a trajetória real percorrida pelo robô em cada execução.

In [43]:
n_runs = len(paths_df)
cols = 2
rows = math.ceil(n_runs / cols)
titles = [f'Run {i+1}' for i in range(n_runs)]

# Calcula o range global de x e y para todos os subplots terem a mesma escala
_all_x = [v for df, plan in zip(paths_df, plans_df)
          for v in list(df['pos_x']) + list(plan['x'])]
_all_y = [v for df, plan in zip(paths_df, plans_df)
          for v in list(df['pos_y']) + list(plan['y'])]
_pad   = 0.2
_x_range = [min(_all_x) - _pad, max(_all_x) + _pad]
_y_range = [min(_all_y) - _pad, max(_all_y) + _pad]

fig = make_subplots(rows=rows, cols=cols,
                    subplot_titles=titles,
                    vertical_spacing=0.04)

for i, (actual_df, last_plan) in enumerate(zip(paths_df, plans_df)):
    row = i // cols + 1
    col = i % cols + 1

    fig.add_trace(go.Scatter(
        x=last_plan['x'], y=last_plan['y'],
        mode='lines', name='Caminho planejado',
        line=dict(color='royalblue', dash='dash', width=2),
        legendgroup='plan', showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=actual_df['pos_x'], y=actual_df['pos_y'],
        mode='lines', name='Trajetória real',
        line=dict(color='tomato', width=2),
        legendgroup='runs', showlegend=(i == 0)
    ), row=row, col=col)

fig.update_xaxes(title_text='X (m)', range=_x_range)
fig.update_yaxes(title_text='Y (m)', range=_y_range)
fig.update_layout(
    title='Caminho Planejado vs Trajetória Real — Todas as Runs',
    hovermode='closest',
    height=400 * rows,
    width=1100,
    legend=dict(
        x=0.02, y=0.98, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='black', borderwidth=1,
    ),
)
apply_ieee_style(fig)
save_fig(fig, 'planned_vs_actual_paths.svg')
fig.show()


#### 2.1 Normalização e Mediana dos Caminhos

Para comparar os caminhos entre diferentes runs, é necessário normalizá-los. Cada execução possui um número diferente de amostras e uma velocidade diferente, então não é possível comparar ponto a ponto diretamente.

A abordagem utilizada é a **parametrização por comprimento de arco**:
1. **Winsorize** — remove poses extremas 
2. **Comprimento de arco normalizado** — mapeia cada ponto para `[0, 1]` proporcionalmente à distância percorrida
3. **Interpolação** — reamostrar todos os caminhos para `n_points` pontos uniformes

Com isso, todos os caminhos ficam no mesmo espaço e é possível calcular a **mediana ponto a ponto** para os dfs de 1 e 2 câmeras.

##### 2.1.1 Caminhos Planejados para duas câmeras (Plans)

Aplicamos a normalização nos caminhos planejados pelo NAV2. Como os planos são gerados com amostras equidistantes, o efeito da interpolação é sutil, mas é necessário para manter o mesmo pipeline dos caminhos reais.

In [44]:
# Parametriza uma trajetória 2D pelo comprimento de arco acumulado (em metros).
# Permite comparar trajetórias com densidades de pontos diferentes,
# pois o parâmetro representa distância física percorrida, não índice nem tempo.
def arc_length_parametrize(pos_x, pos_y):
    coords = np.column_stack([pos_x, pos_y])
    deltas = np.diff(coords, axis=0) # diferença entre pontos consecutivos
    arc = np.concatenate([[0], np.cumsum(np.hypot(deltas[:, 0], deltas[:, 1]))])

    if DEBUG:
        print(f"[arc_length_parametrize]")
        print(f"  n_points     : {len(pos_x)}")
        print(f"  total length : {arc[-1]:.4f} m")
        print(f"  arc[:5]  : {arc[:5]}")
        print(f"  arc[-5:] : {arc[-5:]}")

    return arc

def plot_interpolation_check(u, pos_x, pos_y, t, fx, fy, title='Interpolation check'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(u, pos_x, 'o', t, fx(t), '-')
    ax1.set_title('X vs arc')
    ax1.set_xlabel('arc (m)')
    ax2.plot(u, pos_y, 'o', t, fy(t), '-')
    ax2.set_title('Y vs arc')
    ax2.set_xlabel('arc (m)')
    plt.suptitle(f'[DEBUG] {title}')
    plt.tight_layout()
    plt.show()

def winsorize_and_resample(df, x_col='pos_x', y_col='pos_y', limits=(0.0075, 0.015), n_points=500, idx=None):
    if idx is not None and idx >= 10:
        if DEBUG:
            print(f"[winsorize_and_resample — plans] skipped run {idx} (single camera — will process later)")
        return None

    pos_x = np.array(winsorize(df[x_col], limits=limits))
    pos_y = np.array(winsorize(df[y_col], limits=limits))

    if DEBUG:
        print(f"[winsorize_and_resample — plans]")
        print(f"  input rows    : {len(df)}")
        print(f"  limits        : {limits}")
        print(f"  pos_x range   : [{pos_x.min():.4f}, {pos_x.max():.4f}]")
        print(f"  pos_y range   : [{pos_y.min():.4f}, {pos_y.max():.4f}]")

    u = arc_length_parametrize(pos_x, pos_y)

    t = np.linspace(0, u[-1], n_points)
    fx = interp1d(u, pos_x, kind='linear')
    fy = interp1d(u, pos_y, kind='linear')

    # Para plotar o gráfico (X, Parâmetro de Arco)
    if DEBUG:
        print(f"  resampled to  : {n_points} points")
        plot_interpolation_check(u, pos_x, pos_y, t, fx, fy, title='Plans')

    return fx(t), fy(t)

# Caminhos planejados (usa colunas x/y)
plans_resampled_2cam = [winsorize_and_resample(df, x_col='x', y_col='y', idx=i) for i, df in enumerate(plans_df)]

plans_xs_2cam = np.array([p[0] for p in plans_resampled_2cam if p is not None])
plans_ys_2cam = np.array([p[1] for p in plans_resampled_2cam if p is not None])
median_plan_x_2cam = np.median(plans_xs_2cam, axis=0)
median_plan_y_2cam = np.median(plans_ys_2cam, axis=0)


**Verificação — Planos 2 câmeras:** comparação entre dados brutos e após reamostagem por comprimento de arco.

In [45]:
_df = plans_df[0]
_raw_x = _df['x'].values
_raw_y = _df['y'].values

_wins_x = np.array(winsorize(_df['x'], limits=(0.0075, 0.015)))
_wins_y = np.array(winsorize(_df['y'], limits=(0.0075, 0.015)))

_interp_x, _interp_y = plans_resampled_2cam[0]

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Sem processamento",
    "Winsorize",
    "Winsorize + Interpolação"
])

fig.add_trace(go.Scatter(
    x=_raw_x, y=_raw_y,
    mode='markers', name='Raw',
    marker=dict(color='gray', size=4)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=_wins_x, y=_wins_y,
    mode='markers', name='Winsorize',
    marker=dict(color='tomato', size=4)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=_interp_x, y=_interp_y,
    mode='markers', name='Winsorize + Interpolação',
    marker=dict(color='royalblue', size=4)
), row=1, col=3)

fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_yaxes(scaleanchor='x3', row=1, col=3)
fig.update_layout(title='Plano 1 — Etapas de Processamento', hovermode='closest', width=1300)
apply_ieee_style(fig)
save_fig(fig, 'plan1_processing_steps.svg')
fig.show()

**Planos 2 câmeras — todos os planos reamostrados** com mediana calculada ponto a ponto.

In [46]:
fig = go.Figure()

for i, (px, py) in enumerate(p for p in plans_resampled_2cam if p is not None):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name=f'Plans',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_plan_x_2cam, y=median_plan_y_2cam,
    mode='lines', name='Median plan',
    line=dict(color='royalblue', dash='dash', width=3)
))

fig.update_layout(
    title='Plans + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
apply_ieee_style(fig)
save_fig(fig, 'plans_2cam_median.svg')
fig.show()

##### 2.1.2 Trajetórias Reais para duas câmeras (Paths)

Aplicamos o mesmo pipeline nos caminhos reais percorridos pelo robô. Aqui o winsorize é acabou nem sendo usado, pela cautela de fazer o robo começar no mesmo ponto.

In [47]:
# Trajetórias reais — duas câmeras (winsorize + interpolação)
def _process_path_winsorize(df, n_points=500):
    pos_x = np.array(winsorize(df['pos_x'], limits=(0.0, 0.00)))
    pos_y = np.array(winsorize(df['pos_y'], limits=(0.0, 0.00)))
    u = arc_length_parametrize(pos_x, pos_y)
    t = np.linspace(0, u[-1], n_points)
    return interp1d(u, pos_x)(t), interp1d(u, pos_y)(t)

paths_winsorized_2cam = [_process_path_winsorize(df) for df in paths_df[:10]]
paths_xs_winsorized_2cam = np.array([p[0] for p in paths_winsorized_2cam])
paths_ys_winsorized_2cam = np.array([p[1] for p in paths_winsorized_2cam])
median_path_winsorized_x_2cam = np.median(paths_xs_winsorized_2cam, axis=0)
median_path_winsorized_y_2cam = np.median(paths_ys_winsorized_2cam, axis=0)


**Verificação — Trajetórias 2 câmeras (winsorize):** comparação entre sem processamento, winsorize e após interpolação.

In [48]:
_df = paths_df[0]
_raw_x = _df['pos_x'].values
_raw_y = _df['pos_y'].values

_wins_x = np.array(winsorize(_df['pos_x'], limits=(0.0, 0.00)))
_wins_y = np.array(winsorize(_df['pos_y'], limits=(0.0, 0.00)))

_interp_x, _interp_y = paths_winsorized_2cam[0]

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Sem processamento",
    "Winsorize",
    "Winsorize + Interpolação"
])

fig.add_trace(go.Scatter(
    x=_raw_x, y=_raw_y,
    mode='markers', name='Raw',
    marker=dict(color='gray', size=4)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=_wins_x, y=_wins_y,
    mode='markers', name='Winsorize',
    marker=dict(color='tomato', size=4)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=_interp_x, y=_interp_y,
    mode='markers', name='Winsorize + Interpolação',
    marker=dict(color='royalblue', size=4)
), row=1, col=3)

fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_yaxes(scaleanchor='x3', row=1, col=3)
fig.update_layout(title='Trajetória 1 — Etapas de Processamento', hovermode='closest', width=1500)
apply_ieee_style(fig)
save_fig(fig, 'path1_processing_steps.svg')
fig.show()

**Trajetórias 2 câmeras (winsorize) — todas as runs** com mediana calculada.

In [49]:
fig = go.Figure()

for i, (px, py) in enumerate(p for p in paths_winsorized_2cam if p is not None):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name=f'Runs',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_2cam, y=median_path_winsorized_y_2cam,
    mode='lines', name='Median path',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Paths + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
apply_ieee_style(fig)
save_fig(fig, 'paths_2cam_median.svg')
fig.show()

**Duas câmeras — mediana final:** sobreposição da mediana do plano calculado com a mediana da trajetória real.

In [50]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_plan_x_2cam, y=median_plan_y_2cam,
    mode="lines", name="Mediana do plano calculado",
    line=dict(color="royalblue", dash="dash", width=2.5)
))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_2cam, y=median_path_winsorized_y_2cam,
    mode="lines", name="Mediana do caminho feito",
    line=dict(color="tomato", width=2.5)
))

fig.update_layout(
    title="Duas Câmeras — Mediana do Plano vs Mediana da Trajetória",
    xaxis_title="X (m)", yaxis_title="Y (m)",
    yaxis_scaleanchor="x", hovermode="closest"
)
apply_ieee_style(fig)
save_fig(fig, 'median_comparison_2cam.svg')
fig.show()

##### MSE — Duas Câmeras

Desvio entre a **mediana das trajetórias reais** e a **mediana dos planos calculados**. Um único número que resume o quanto o robô desviou do plano em média ao longo do percurso.

In [51]:
from sklearn.metrics import mean_squared_error

mse_x = mean_squared_error(median_plan_x_2cam, median_path_winsorized_x_2cam)
mse_y = mean_squared_error(median_plan_y_2cam, median_path_winsorized_y_2cam)
mse_2cam  = (mse_x + mse_y) / 2
rmse_2cam = np.sqrt(mse_2cam)
print(f'MSE  (mediana trajetória vs mediana plano) — duas câmeras: {mse_2cam:.6f} m²')
print(f'RMSE (mediana trajetória vs mediana plano) — duas câmeras: {rmse_2cam:.4f} m')


MSE  (mediana trajetória vs mediana plano) — duas câmeras: 0.012590 m²
RMSE (mediana trajetória vs mediana plano) — duas câmeras: 0.1122 m


#### 2.2 Uma Câmera (Runs 11–20)

Repetimos o mesmo pipeline para as 10 execuções com uma câmera. As funções são idênticas — apenas o conjunto de dados muda.

##### 2.2.1 Caminhos Planejados

In [52]:
plans_resampled_1cam = [winsorize_and_resample(df, x_col='x', y_col='y') for df in plans_df[10:]]

plans_xs_1cam = np.array([p[0] for p in plans_resampled_1cam if p is not None])
plans_ys_1cam = np.array([p[1] for p in plans_resampled_1cam if p is not None])
median_plan_x_1cam = np.median(plans_xs_1cam, axis=0)
median_plan_y_1cam = np.median(plans_ys_1cam, axis=0)

In [53]:
fig = go.Figure()

for i, (px, py) in enumerate(p for p in plans_resampled_1cam if p is not None):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name='Plans',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='plans', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_plan_x_1cam, y=median_plan_y_1cam,
    mode='lines', name='Mediana dos planos',
    line=dict(color='royalblue', dash='dash', width=3)
))

fig.update_layout(
    title='Planos — Uma Câmera + Mediana',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
apply_ieee_style(fig)
save_fig(fig, 'plans_1cam_median.svg')
fig.show()

##### 2.2.2 Trajetórias Reais para uma câmera (Paths)

Aplicamos exatamente o mesmo pipeline das duas câmeras (winsorize + interpolação por comprimento de arco). Anteriormente esta seção usava um *clip pela geometria do plano* como alternativa, mas a nova coleta de dados está limpa o suficiente para que o winsorize sozinho seja suficiente.

In [54]:
# Trajetórias reais — uma câmera (winsorize + interpolação)
paths_winsorized_1cam = [_process_path_winsorize(df) for df in paths_df[10:]]
paths_xs_winsorized_1cam = np.array([p[0] for p in paths_winsorized_1cam])
paths_ys_winsorized_1cam = np.array([p[1] for p in paths_winsorized_1cam])
median_path_winsorized_x_1cam = np.median(paths_xs_winsorized_1cam, axis=0)
median_path_winsorized_y_1cam = np.median(paths_ys_winsorized_1cam, axis=0)


Verificação visual da interpolação: o gráfico abaixo mostra todos os caminhos e a mediana calculada a partir das etapas de preprocessamentos citadas anteriormente.

In [55]:
fig = go.Figure()

for i, (rx, ry) in enumerate(p for p in paths_winsorized_1cam if p is not None):
    fig.add_trace(go.Scatter(
        x=rx, y=ry,
        mode='lines', name='Runs',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_1cam, y=median_path_winsorized_y_1cam,
    mode='lines', name='Mediana das trajetórias',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Trajetórias — Uma Câmera (Winsorize) + Mediana',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
apply_ieee_style(fig)
save_fig(fig, 'paths_1cam_median.svg')
fig.show()

Por fim será demonstrado gráficamente a junção das medianas tanto do caminho calculado, quanto o caminho feito.

In [56]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_plan_x_1cam, y=median_plan_y_1cam,
    mode="lines", name="Mediana do plano calculado",
    line=dict(color="royalblue", dash="dash", width=2.5)
))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_1cam, y=median_path_winsorized_y_1cam,
    mode="lines", name="Mediana do caminho feito",
    line=dict(color="tomato", width=2.5)
))

fig.update_layout(
    title="Uma Câmera — Mediana do Plano vs Mediana da Trajetória",
    xaxis_title="X (m)", yaxis_title="Y (m)",
    yaxis_scaleanchor="x", hovermode="closest"
)
apply_ieee_style(fig)
save_fig(fig, 'median_comparison_1cam.svg')
fig.show()

##### MSE — Uma Câmera

Desvio entre a **mediana das trajetórias reais** e a **mediana dos planos calculados** para as runs com uma câmera.

In [57]:
from sklearn.metrics import mean_squared_error

mse_x = mean_squared_error(median_plan_x_1cam, median_path_winsorized_x_1cam)
mse_y = mean_squared_error(median_plan_y_1cam, median_path_winsorized_y_1cam)
mse_1cam  = (mse_x + mse_y) / 2
rmse_1cam = np.sqrt(mse_1cam)
print(f'MSE  (mediana trajetória vs mediana plano) — uma câmera: {mse_1cam:.6f} m²')
print(f'RMSE (mediana trajetória vs mediana plano) — uma câmera: {rmse_1cam:.4f} m')

MSE  (mediana trajetória vs mediana plano) — uma câmera: 0.012198 m²
RMSE (mediana trajetória vs mediana plano) — uma câmera: 0.1104 m


#### 2.3 Comparação entre Grupos

Análise comparativa entre os dois grupos de execução — robô com duas câmeras (runs 1–10) e com uma câmera (runs 11–20) — dividida em duas etapas: análise gráfica das trajetórias e análise quantitativa via MSE/RMSE.

##### 2.3.1 Análise Gráfica

Comparação visual das trajetórias medianas dos dois grupos em relação ao plano mediano global.

In [58]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Duas Câmeras', 'Uma Câmera'])

# Duas câmeras
fig.add_trace(go.Scatter(
    x=median_plan_x_2cam, y=median_plan_y_2cam,
    mode='lines', name='Plano mediano',
    line=dict(color='gray', dash='dash', width=2),
    legendgroup='plan', showlegend=True
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_2cam, y=median_path_winsorized_y_2cam,
    mode='lines', name='Mediana trajetória',
    line=dict(color='royalblue', width=2.5),
    legendgroup='path', showlegend=True
), row=1, col=1)

# Uma câmera
fig.add_trace(go.Scatter(
    x=median_plan_x_1cam, y=median_plan_y_1cam,
    mode='lines', name='Plano mediano',
    line=dict(color='gray', dash='dash', width=2),
    legendgroup='plan', showlegend=True
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_1cam, y=median_path_winsorized_y_1cam,
    mode='lines', name='Mediana trajetória',
    line=dict(color='tomato', width=2.5),
    legendgroup='path', showlegend=True
), row=1, col=2)

fig.update_yaxes(scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', row=1, col=1)
fig.update_layout(
    title='Comparação: Mediana do Plano vs Mediana da Trajetória',
    hovermode='closest', width=1100
)
apply_ieee_style(fig)
save_fig(fig, 'trajectory_comparison.svg')
fig.show()

Para a comparação final, calcula-se a **mediana global do caminho planejado** — combinando os planos de ambos os grupos (duas e uma câmera). Com isso é possível visualizar num único gráfico o quanto cada grupo desviou em relação ao mesmo plano de referência.

In [59]:
# Mediana global do plano (une os dois grupos)
all_plans_xs = np.vstack([plans_xs_2cam, plans_xs_1cam])
all_plans_ys = np.vstack([plans_ys_2cam, plans_ys_1cam])
median_plan_x_global = np.median(all_plans_xs, axis=0)
median_plan_y_global = np.median(all_plans_ys, axis=0)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_plan_x_global, y=median_plan_y_global,
    mode='lines', name='Plano mediano (global)',
    line=dict(color='gray', dash='dash', width=2)
))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_2cam, y=median_path_winsorized_y_2cam,
    mode='lines', name='Mediana trajetória — duas câmeras',
    line=dict(color='royalblue', width=2.5)
))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_1cam, y=median_path_winsorized_y_1cam,
    mode='lines', name='Mediana trajetória — uma câmera',
    line=dict(color='tomato', width=2.5)
))

fig.update_layout(
    title='Plano Mediano Global vs Medianas das Trajetórias',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest',
    height=720, width=920,
    legend=dict(
        x=0.02, y=0.98, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='black', borderwidth=1,
    ),
)


fig.update_xaxes(range=[-2.2, 2.0])
fig.update_yaxes(range=[-1.6, 1.8])
apply_ieee_style(fig)
save_fig(fig, 'global_plan_vs_trajectories.svg')
fig.show()

Como podemos ver, os dois caminhos ficaram bem parecidos, o que acabei ficando um pouco surpreso. Uma análise que pode nos dar uma informação mais precisa é o MSE e RMSE que calculamos anteriormente para os dois casos, esses valores quantificam exatamente o desvio médio de cada grupo em relação ao plano, independentemente da similaridade visual.

In [60]:
print('=' * 45)
print(f'  MSE  — duas câmeras : {mse_2cam:.6f} m²')
print(f'  RMSE — duas câmeras : {rmse_2cam:.4f} m')
print('-' * 45)
print(f'  MSE  — uma câmera   : {mse_1cam:.6f} m²')
print(f'  RMSE — uma câmera   : {rmse_1cam:.4f} m')
print('=' * 45)
diff_rmse = abs(rmse_2cam - rmse_1cam)
better = 'duas câmeras' if rmse_2cam < rmse_1cam else 'uma câmera'
print(f'  Diferença RMSE      : {diff_rmse:.4f} m')
print(f'  Grupo mais preciso  : {better}')
print('=' * 45)

  MSE  — duas câmeras : 0.012590 m²
  RMSE — duas câmeras : 0.1122 m
---------------------------------------------
  MSE  — uma câmera   : 0.012198 m²
  RMSE — uma câmera   : 0.1104 m
  Diferença RMSE      : 0.0018 m
  Grupo mais preciso  : uma câmera


##### 2.3.2 Análise Quantitativa — MSE e RMSE

O MSE e RMSE medem numericamente o desvio médio de cada grupo em relação ao plano. O boxplot complementa mostrando a dispersão dos erros entre as runs individuais — revelando se o desvio é consistente ou se há execuções muito discrepantes.

In [61]:
# RMSE de cada run individualmente vs plano mediano global
rmse_runs_2cam, runs_2cam = [], []
for i, (rx, ry) in enumerate(p for p in paths_winsorized_2cam if p is not None):
    d = (rx - median_plan_x_global)**2 + (ry - median_plan_y_global)**2
    rmse_runs_2cam.append(np.sqrt(np.mean(d)))
    runs_2cam.append(i + 1)        # runs 1..10

rmse_runs_1cam, runs_1cam = [], []
for i, (rx, ry) in enumerate(p for p in paths_winsorized_1cam if p is not None):
    d = (rx - median_plan_x_global)**2 + (ry - median_plan_y_global)**2
    rmse_runs_1cam.append(np.sqrt(np.mean(d)))
    runs_1cam.append(i + 11)       # runs 11..20
    
fig = go.Figure()
fig.add_trace(go.Box(
    y=rmse_runs_2cam, name='Duas câmeras',
    marker_color='royalblue', boxpoints='outliers'
))
fig.add_trace(go.Box(
    y=rmse_runs_1cam, name='Uma câmera',
    marker_color='tomato', boxpoints='outliers'
))
fig.update_layout(
    title='Dispersão do RMSE por Run — Duas Câmeras vs Uma Câmera',
    yaxis_title='RMSE (m)', hovermode='closest'
)
apply_ieee_style(fig)
save_fig(fig, 'rmse_boxplot.svg')
fig.show()

**Discussão - boxplot e conclusão sobre a equivalência das trajetórias**

O boxplot é especialmente útil aqui porque dá ao mesmo tempo uma visão da **distribuição do RMSE por run** (mediana, IQR e variação dos valores) e expõe explicitamente os **outliers**, pontos que ficam fora das cercas de Tukey (Q1 − 1.5·IQR, Q3 + 1.5·IQR).

Olhando os pontos rotulados, fica claro que existe **um outlier puxando o RMSE de duas câmeras para cima: a Run 6**. Ao inspecionar esse run em particular, percebe-se que ela teve uma falha de processamento, a localização só começou a ser contabilizada depois de algum tempo do início da execução, deixando uma faixa inicial sem dados e penalizando o RMSE médio dessa run especificamente. Não é, portanto, um reflexo da qualidade do sistema com duas câmeras, e sim de um problema isolado de coleta.

Mesmo com a Run 6 inflando o grupo de duas câmeras, o RMSE final ficou **muito próximo entre os dois modos**, diferença de apenas **0.0018 m** (ver a comparação MSE/RMSE acima). Combinando essa proximidade quantitativa com a sobreposição visual das medianas das trajetórias na análise gráfica, **conclui-se que o desempenho de localização com uma e com duas câmeras foi equivalente** neste benchmark.

A análise gráfica e quantitativa (MSE/RMSE) das trajetórias indicou desempenho **equivalente** entre os dois modos. No entanto, "equivalência na trajetória final" não garante que os dois modos cheguem lá da mesma forma, é possível que um deles esteja se localizando com mais incerteza, fazendo mais loop closures, ou consumindo mais recursos para entregar a mesma qualidade de caminho.

Por isso, a análise continua a seguir com foco nas **métricas internas de localização do RTABMAP** (`inlier_ratio`, `hypothesis_ratio`, `cov_pos_trace`, `detection_time_ms`, etc.), buscando identificar diferenças entre o uso de uma e duas câmeras que não aparecem na análise gráfica e no RMSE.

### 3. Qualidade da Localização

Análise das métricas internas do RTABMAP para cada grupo. Os dados são agrupados por câmera e comparados via boxplot (quando possível) para revelar diferenças na qualidade de localização visual, confiança da estimativa e custo computacional. As métricas de localização disponíveis são:

**Qualidade do casamento visual (RANSAC / odometria visual):**
- **`inliers`** — número de correspondências de features consideradas geometricamente consistentes pelo RANSAC ao estimar a transformação entre frames. Valores altos indicam que a cena tem textura suficiente e que o casamento visual foi bem-sucedido.
- **`matches`** — total de correspondências de features encontradas antes da filtragem geométrica. Reflete a riqueza visual da cena.
- **`inlier_ratio`** = `inliers / words(frame_atual)` — fração dos inliers RANSAC em relação ao total de palavras visuais (BoW) do frame atual (`toSignature.getWords().size()` em `RegistrationVis.cpp`). Valores baixos sugerem cenas pouco texturizadas, oclusões ou movimento brusco.

**Confiança do reconhecimento de lugar (loop closure):**
- **`hypothesis_ratio`** = `melhor_hipótese / loop_closure_hipótese` — razão entre o valor da melhor hipótese corrente e o valor da hipótese que gerou o último loop closure aceito (em `Rtabmap.cpp`). Valor == 1 indica que a hipótese atual é tão confiante quanto o loop closure aceito; valores baixos indicam menor confiança no reconhecimento de lugar.
- **`loop_closure_id`** — id do nó com o qual ocorreu o fechamento de laço (`-1` quando não houve). Útil para contar eventos de relocalização ao longo da trajetória.

**Incerteza da pose estimada:**
- **`cov_xx`, `cov_yy`, `cov_yaw`** — variâncias diagonais da matriz de covariância da pose. Quanto menores, mais confiante está o filtro sobre a estimativa.
- **`cov_pos_trace`** = `cov_xx + cov_yy` — traço da submatriz de posição; resume em um único escalar a incerteza translacional da estimativa.

**Custo computacional e gerenciamento de memória:**
- **`detection_time_ms`** — tempo gasto pelo módulo de detecção de loop closure por atualização.
- **`total_time_ms`** — tempo total de processamento do RTABMAP por atualização (detecção + manutenção do grafo + memória).
- **`wm_size`** — tamanho da *Working Memory* (número de nós ativos do grafo). Cresce com a exploração e impacta diretamente o custo computacional.

#### 3.1 Preparação dos Dados

Agregamos todos os logs de localização separados por grupo para facilitar a comparação.

In [62]:
# Concatena todos os runs de cada grupo com label
loc_2cam_raw = pd.concat(
    [df.assign(run=i+1) for i, df in enumerate(runs_loc_df[:10])],
    ignore_index=True
)
loc_1cam_raw = pd.concat(
    [df.assign(run=i+11) for i, df in enumerate(runs_loc_df[10:])],
    ignore_index=True
)

Plot do histograma para ver a distribuição dos dados e ter uma ideia de como estão, se possuem outliers.

In [63]:
metrics = ['inliers', 'inlier_ratio', 'hypothesis_ratio', 'cov_pos_trace', 'detection_time_ms']

fig = make_subplots(
    rows=len(metrics), cols=2,
    column_titles=['Duas Câmeras', 'Uma Câmera'],
    row_titles=metrics,
    vertical_spacing=0.06
)

for row, metric in enumerate(metrics, start=1):
    fig.add_trace(go.Histogram(
        x=loc_2cam_raw[metric], name=metric,
        marker_color='royalblue', opacity=0.7,
        showlegend=False
    ), row=row, col=1)
    fig.add_trace(go.Histogram(
        x=loc_1cam_raw[metric], name=metric,
        marker_color='tomato', opacity=0.7,
        showlegend=False
    ), row=row, col=2)

fig.update_layout(
    title='Distribuição das Métricas — Duas Câmeras vs Uma Câmera',
    height=300 * len(metrics), width=900,
    hovermode='closest'
)
apply_ieee_style(fig)
save_fig(fig, 'metrics_histograms.svg')
fig.show()

Observando os histogramas, é nítida a presença de outliers apenas uma métrica agora: `inlier_ratio`. A seguir é feita uma análise para tentar explicar esses outliers e avaliar se é possível removê-los ou se representam alguma informação relevante sobre a localização durante a trajetória do robô.

**Hipótese 1 — `inlier_ratio = 0`:** correspondem a momentos em que há perda de frames das câmeras ou ao início da run, com o robô ainda parado, em que o RTABMAP ainda não processa correspondências visuais.

Também é perceptível que a `covariância > 100` não está presente nesse *dataset*.

Para verificar essa  hipótese, plotamos abaixo `inlier_ratio` em função do tempo relativo ao início de cada run.

In [64]:
# Análise temporal dos outliers — usa os dados brutos (sem filtro) para
# revelar EM QUE INSTANTE da trajetória os valores extremos aparecem.
def _concat_with_rel_time(runs_subset, run_offset):
    parts = []
    for i, df in enumerate(runs_subset):
        d = df.copy()
        d['run']   = i + run_offset
        d['t_rel'] = d['timestamp_sec'] - d['timestamp_sec'].min()
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

_loc_2cam_t = _concat_with_rel_time(runs_loc_df[:10], 1)
_loc_1cam_t = _concat_with_rel_time(runs_loc_df[10:], 11)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'inlier_ratio — Duas Câmeras', 'inlier_ratio — Uma Câmera'
    ),
    vertical_spacing=0.13, horizontal_spacing=0.08,
)

for df_g, color, col in [(_loc_2cam_t, 'royalblue', 1),
                         (_loc_1cam_t, 'tomato',    2)]:
    for run_id, df_run in df_g.groupby('run'):
        fig.add_trace(go.Scatter(
            x=df_run['t_rel'], y=df_run['inlier_ratio'],
            mode='markers', marker=dict(color=color, size=5),
            opacity=0.7, showlegend=False,
            hovertemplate=f'run {run_id}<br>t=%{{x:.1f}}s<br>inlier_ratio=%{{y:.2f}}<extra></extra>'
        ), row=1, col=col)

for col in (1, 2):
    fig.update_xaxes(title_text='Tempo relativo ao início da run (s)', row=1, col=col)
fig.update_yaxes(title_text='inlier_ratio',        row=1, col=1)

fig.update_layout(
    title='Outliers ao longo da trajetória — inlier_ratio e cov_pos_trace vs. tempo',
    height=720, width=1600, hovermode='closest',
)

fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

apply_ieee_style(fig)
save_fig(fig, 'temporal_outliers_inlier_ratio.svg')
fig.show()

##### Análise detalhada da Run 9 — `inlier_ratio = 0` no final da trajetória

O plot anterior reforça a hipótese de que os `inlier_ratio = 0` ocorrem predominantemente **no início das runs** (aquecimento do sistema de localização) e em **momentos isolados de perda de frames** da própria câmera.

No entanto, um caso chamou a atenção: a **Run 9**, que foi a run de maior duração (~63 segundos). A partir dos ~29 s, o `inlier_ratio` permanece zerado continuamente até o final da bag — um padrão bem diferente dos demais. A hipótese mais provável é que o robô chegou à linha de chegada e parou, mas o **gravador da bag não foi interrompido**, fazendo com que o sistema continuasse a gravar a localização do robô com ele parado.

A seguir, a Run 9 é analisada isoladamente para entender todos os outliers relacionados ao `inlier_ratio = 0`.

In [65]:
def plot_run_trajectory(run_df, checkpoint_interval=10, run_label=None, save_path=None):
    """Plota a trajetória de uma run com checkpoints temporais.

    Args:
        run_df: DataFrame da run (deve ter pos_x, pos_y, timestamp_sec).
        checkpoint_interval: intervalo em segundos entre checkpoints (default 10).
        run_label: rótulo usado no título (ex: 'Run 9'). Inferido do df se omitido.
    """
    df = run_df.copy()
    df['t_rel'] = df['timestamp_sec'] - df['timestamp_sec'].min()

    if run_label is None:
        run_label = f'Run (n={len(df)})'

    t_marks = range(0, int(df['t_rel'].max()) + checkpoint_interval, checkpoint_interval)
    checkpoints = []
    for t in t_marks:
        idx = (df['t_rel'] - t).abs().idxmin()
        checkpoints.append(df.loc[idx])
    checkpoints_df = pd.DataFrame(checkpoints)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df['pos_x'], y=df['pos_y'],
        mode='lines+markers',
        marker=dict(
            color=df['t_rel'],
            colorscale='RdYlBu_r',
            size=6,
            showscale=False,
        ),
        line=dict(color='lightgray', width=1),
        name='trajetória',
        hovertemplate='t=%{customdata:.1f}s<br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>',
        customdata=df['t_rel'],
    ))

    fig.add_trace(go.Scatter(
        x=checkpoints_df['pos_x'], y=checkpoints_df['pos_y'],
        mode='markers+text',
        marker=dict(color='black', size=14, symbol='circle'),
        text=[f'{t:.0f}s' for t in checkpoints_df['t_rel']],
        textposition='top center',
        textfont=dict(size=16),
        name=f'checkpoint ({checkpoint_interval}s)',
        hovertemplate='t=%{customdata:.1f}s<br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>',
        customdata=checkpoints_df['t_rel'],
    ))

    # for label, row, color, sym in [
    #     ('início', df.iloc[0],  'green', 'triangle-up'),
    #     ('fim',    df.iloc[-1], 'red',   'square'),
    # ]:
    #     fig.add_trace(go.Scatter(
    #         x=[row['pos_x']], y=[row['pos_y']],
    #         mode='markers+text',
    #         marker=dict(color=color, size=14, symbol=sym),
    #         text=[label], textposition='bottom center',
    #         textfont=dict(size=12, color=color),
    #         name=label, showlegend=True,
    #     ))

    fig.update_layout(
        title=f'{run_label} — Trajetória com checkpoints temporais (a cada {checkpoint_interval}s)',
        xaxis_title='X (m)', yaxis_title='Y (m)',
        yaxis_scaleanchor='x', hovermode='closest',
        height=720, width=920,
        legend=dict(
            x=0.02, y=0.98, xanchor='left', yanchor='top',
            bgcolor='white', bordercolor='black', borderwidth=1,
        ),
    )
    apply_ieee_style(fig)
    if save_path:
        fig.write_image(save_path)
    fig.show()


# Run 9 com checkpoints a cada 10 s
_run9_path = runs_loc_df[8].copy()
plot_run_trajectory(_run9_path, checkpoint_interval=1, run_label='Run 9', save_path='graphs/run9_trajectory.svg')

**Conclusão — Run 9:** a hipótese se confirma. O robô chegou à linha de chegada por volta dos **~26.7 s** e permaneceu parado até o encerramento da bag (~63 s), pois o gravador não foi interrompido a tempo.

As pequenas movimentações visíveis no gráfico após esse ponto (perceptíveis ao dar zoom na região da linha de chegada) **não representam deslocamento real** do robô. Trata-se do **drift da odometria** acumulado com o robô estático. O Go2 utiliza a IMU como uma das fontes de odometria, e a IMU integra continuamente a aceleração para estimar posição; qualquer inclinação residual do robô sobre o solo gera uma componente de gravidade que, após dupla integração, produz exatamente esse tipo de drift.

Como tudo que está após ~26.7 s não representa localização válida durante a corrida, **esses frames serão removidos da Run 9** antes de continuar a análise comparativa das métricas do RTABMAP.

In [66]:
RUN_9_CUTOFF = 26.7   # s a partir do qual o robô ficou parado na linha de chegada

_run9_raw = runs_loc_df[8].copy()
_run9_raw['t_rel'] = _run9_raw['timestamp_sec'] - _run9_raw['timestamp_sec'].min()

run9_clean = _run9_raw[_run9_raw['t_rel'] <= RUN_9_CUTOFF].copy()

print(f'Run 9 — bruto   : {len(_run9_raw):3d} frames  (t_rel máx = {_run9_raw["t_rel"].max():.1f} s)')
print(f'Run 9 — limpo   : {len(run9_clean):3d} frames  (t_rel máx = {run9_clean["t_rel"].max():.1f} s)')
print(f'Frames removidos: {len(_run9_raw) - len(run9_clean)}')

# Substitui a run 9 (índice 8) na lista global de runs
runs_loc_df_clean = runs_loc_df.copy()
runs_loc_df_clean[8] = run9_clean

# Plotando novamente a run 9 para verificar que foi removido os frames inválidos
plot_run_trajectory(runs_loc_df_clean[8], checkpoint_interval=1, run_label='Run 9 Depois do CutOff')

Run 9 — bruto   :  57 frames  (t_rel máx = 63.1 s)
Run 9 — limpo   :  24 frames  (t_rel máx = 26.7 s)
Frames removidos: 33


##### Tratamento dos `inlier_ratio = 0` — separação por contexto

Com a Run 9 corrigida, os DataFrames agregados são reconstruídos e os frames com `inlier_ratio = 0` são tratados de forma diferenciada, divididos em duas categorias:

- **Startup** (`t_rel < 4 s`): frames nos primeiros 4 segundos de cada run, quando o robô ainda está parado e o RTABMAP ainda não processou correspondências visuais. São descartados por serem um artefato de inicialização, sem valor analítico.
- **Falhas durante a run** (`t_rel ≥ 4 s`, `inlier_ratio = 0`): frames em que o sistema falhou em encontrar correspondências visuais enquanto o robô já estava em movimento. Esses frames são **removidos das métricas de qualidade** mas **armazenados separadamente**. A frequência dessas falhas é em si uma métrica de robustez, e será usada a seguir para comparar qual configuração (uma ou duas câmeras) sofreu maior perda de rastreamento ao longo das runs.

In [67]:
STARTUP_THRESHOLD = 4.0


# Adiciona uma nova coluna chamada 't_rel' em cada DataFrame do dataset
def _add_t_rel(df):
    d = df.copy()
    d['t_rel'] = d['timestamp_sec'] - d['timestamp_sec'].min()
    return d

# Separa em 3 categorias:
#   startup   -> inlier_ratio=0 AND t_rel < STARTUP_THRESHOLD  (descartado)
#   failures  -> inlier_ratio=0 AND t_rel >= STARTUP_THRESHOLD (guardado para análise de robustez)
#   valid     -> inlier_ratio > 0                              (usado nas métricas de qualidade)
def _categorize(df):
    is_zero  = df['inlier_ratio'] == 0
    startup  = df[ is_zero & (df['t_rel'] <  STARTUP_THRESHOLD)]
    failures = df[ is_zero & (df['t_rel'] >= STARTUP_THRESHOLD)]
    valid    = df[~is_zero]
    return startup, failures, valid

# Taxa de falha por run (excluindo startup)
#
# O groupby('run') divide o DataFrame em sub-DataFrames, um por run.
# A cada iteração o for desempacota dois valores:
#   run_id -> valor da coluna 'run' (1, 2, 3 ... 10 para duas câmeras)
#   grp    -> sub-DataFrame com TODAS as linhas daquela run
#
# Exemplo visual do que o groupby entrega a cada iteração:
#
#   run_id = 1
#   grp:
#     timestamp_sec  camera_mode  node_id  inliers
#     1234.5           Single      1926       1
#     1237.0           Single      1980       1
#     1239.5           Single      1995       1
#     ...
#
# A cada iteração um dicionário {run, pct_fail} é adicionado à lista rows[],
# que no final é convertida em DataFrame via pd.DataFrame(rows).
def _failure_rate_per_run(raw_df, startup_threshold=STARTUP_THRESHOLD):
    rows = []
    for run_id, grp in raw_df.groupby('run'):
        after_startup = grp[grp['t_rel'] >= startup_threshold]
        total  = len(after_startup)
        failed = (after_startup['inlier_ratio'] == 0).sum()
        rows.append({'run': run_id, 'pct_fail': 100 * failed / total})
    return pd.DataFrame(rows)

# Reconstrói os DataFrames agregados com a run 9 corrigida e também adiciona duas novas colunas: t_rel e run
loc_2cam_raw = pd.concat(
    [_add_t_rel(df).assign(run=i+1) for i, df in enumerate(runs_loc_df_clean[:10])],
    ignore_index=True)

loc_1cam_raw = pd.concat(
    [_add_t_rel(df).assign(run=i+11) for i, df in enumerate(runs_loc_df_clean[10:])],
    ignore_index=True)

# Separa em 3 categorias o inlier_ratio == 0 (startup, failures e valid)
startup_2cam, failures_2cam, loc_2cam = _categorize(loc_2cam_raw)
startup_1cam, failures_1cam, loc_1cam = _categorize(loc_1cam_raw)

if DEBUG: 
    print(f'{"Grupo":<16} {"Total":>7} {"Startup":>10} {"Falhas":>13} {"Válidos":>13}')
    print('-' * 68)
    for label, raw, startup, fail, valid in [
        ('Duas câmeras', loc_2cam_raw, startup_2cam, failures_2cam, loc_2cam),
        ('Uma câmera',   loc_1cam_raw, startup_1cam, failures_1cam, loc_1cam)]:
        n = len(raw)
        print(f'{label:<16} {n:>6}  '
            f'{len(startup):>4} ({100*len(startup)/n:4.1f}%)  '
            f'{len(fail):>4} ({100*len(fail)/n:4.1f}%)  '
            f'{len(valid):>4} ({100*len(valid)/n:4.1f}%)')
    print('\n')


# Processa todos os dfs e retorna um df de run por pct_fail
fail_rate_2cam = _failure_rate_per_run(loc_2cam_raw)
fail_rate_1cam = _failure_rate_per_run(loc_1cam_raw)

# Pega a mediana
fail_rate_2cam_median = fail_rate_2cam["pct_fail"].median()
fail_rate_1cam_median = fail_rate_1cam["pct_fail"].median()

print(f'Mediana taxa de falha — duas câmeras : {fail_rate_2cam_median:.1f}%')
print(f'Mediana taxa de falha — uma câmera   : {fail_rate_1cam_median:.1f}%')

# Verificação de qual dos dois testes apresentou maior perda de rastreamento
mais_robusta = 'Duas câmeras' if fail_rate_2cam_median > fail_rate_1cam_median else 'Uma câmera'

print(f'\n-> {mais_robusta} apresentou maior perda de rastreamento.')

Mediana taxa de falha — duas câmeras : 31.4%
Mediana taxa de falha — uma câmera   : 23.5%

-> Duas câmeras apresentou maior perda de rastreamento.


Com os `inlier_ratio = 0` devidamente análisados, os dados válidos estão prontos para a análise de qualidade da localização. A seguir, a distribuição do próprio `inlier_ratio > 0` é examinada comparativamente entre os dois grupos.

In [68]:
# Remove amostras sem localização válida:
#   1) inlier_ratio=0         -> nenhuma correspondência visual encontrada naquele frame ou perda de frame

def _filter(df):
    return df[(df['inlier_ratio']  >  0)].copy()

loc_2cam = _filter(loc_2cam_raw)
loc_1cam = _filter(loc_1cam_raw)

n_removed_2cam = len(loc_2cam_raw) - len(loc_2cam)
n_removed_1cam = len(loc_1cam_raw) - len(loc_1cam)
print(f'Duas câmeras : {len(loc_2cam_raw):5d} amostras brutas → {len(loc_2cam):5d} válidas ({n_removed_2cam:4d} removidas)')
print(f'Uma câmera   : {len(loc_1cam_raw):5d} amostras brutas → {len(loc_1cam):5d} válidas ({n_removed_1cam:4d} removidas)')

Duas câmeras :   222 amostras brutas →   134 válidas (  88 removidas)
Uma câmera   :   220 amostras brutas →   136 válidas (  84 removidas)


#### 3.2 Análise dos dados

#####  3.2.1 Inlier Ratio

Proporção de inliers RANSAC em relação ao total de palavras visuais (BoW) do frame atual (conforme  em `RegistrationVis.cpp`). Valor alto indica que o RTABMAP encontrou muitos pontos geometricamente consistentes com o frame de referência (sinal de odometria visual confiável).

In [69]:
fig = go.Figure()
fig.add_trace(go.Box(
    y=loc_2cam['inlier_ratio'], name='Duas câmeras',
    marker_color='royalblue', boxpoints='outliers'
))
fig.add_trace(go.Box(
    y=loc_1cam['inlier_ratio'], name='Uma câmera',
    marker_color='tomato', boxpoints='outliers'
))
fig.update_layout(
    title='Inlier Ratio — Duas Câmeras vs Uma Câmera',
    yaxis_title='Inlier Ratio', hovermode='closest'
)
apply_ieee_style(fig)
save_fig(fig, 'inlier_ratio_boxplot.svg')
fig.show()

loc_2cam_inlier_ratio_median = loc_2cam['inlier_ratio'].median()
loc_1cam_inlier_ratio_median = loc_1cam['inlier_ratio'].median()

print(f'Mediana inlier_ratio — duas câmeras : {loc_2cam_inlier_ratio_median:.4f}')
print(f'Mediana inlier_ratio — uma câmera   : {loc_1cam_inlier_ratio_median:.4f}')

bigger_inlier = 'Duas câmeras' if loc_2cam_inlier_ratio_median > loc_1cam_inlier_ratio_median else 'Uma câmera'
print(f'\n-> {bigger_inlier} encontrou mais pontos consistentes com o mapa (inlier_ratio)')


Mediana inlier_ratio — duas câmeras : 0.0958
Mediana inlier_ratio — uma câmera   : 0.1480

-> Uma câmera encontrou mais pontos consistentes com o mapa (inlier_ratio)


#####  3.2.2 Hypothesis Ratio

O `hypothesis_ratio` é calculado pelo RTABMAP (`Rtabmap.cpp`) como:

$$\text{hypothesis\_ratio} = \frac{\text{score da hipótese atual}}{\text{score da hipótese do loop closure aceito}}$$

**Interpretação:**
- **Igual a 1** → a hipótese corrente é exatamente o loop closure aceito — confiança máxima de reconhecimento de lugar
- **Igual a 0** → nenhum loop closure ativo como referência — localização ambígua

In [70]:
# Total de frames com hypothesis_ratio = 1 para cada grupo
total_2cam = len(loc_2cam)
total_1cam = len(loc_1cam)
count_one_2cam = (loc_2cam['hypothesis_ratio'] == 1).sum()
count_one_1cam = (loc_1cam['hypothesis_ratio'] == 1).sum()
pct_one_2cam = count_one_2cam / total_2cam * 100
pct_one_1cam = count_one_1cam / total_1cam * 100

colors = ['royalblue', 'tomato']
groups = ['Duas câmeras', 'Uma câmera']

fig = go.Figure(go.Bar(
    x=groups, y=[pct_one_2cam, pct_one_1cam],
    marker_color=colors,
    text=[f'{pct_one_2cam:.1f}%', f'{pct_one_1cam:.1f}%'],
    textposition='outside',
    showlegend=False
))

fig.update_yaxes(title_text='%', range=[0, 100])
fig.update_layout(
    title='Frames com Hypothesis Ratio = 1 (%)',
    hovermode='closest', width=600
)
apply_ieee_style(fig)
save_fig(fig, 'hypothesis_ratio_bar.svg')
fig.show()

print(f'Duas câmeras : {count_one_2cam:4d} / {total_2cam} frames com ratio=1  ({pct_one_2cam:.1f}%)')
print(f'Uma câmera   : {count_one_1cam:4d} / {total_1cam} frames com ratio=1  ({pct_one_1cam:.1f}%)')

melhor = 'Duas câmeras' if pct_one_2cam > pct_one_1cam else 'Uma câmera'
print(f'\n -> {melhor} obteve hipótese confirmada em maior proporção dos frames.')


Duas câmeras :   95 / 134 frames com ratio=1  (70.9%)
Uma câmera   :   62 / 136 frames com ratio=1  (45.6%)

 -> Duas câmeras obteve hipótese confirmada em maior proporção dos frames.


#####  3.2.3 Covariância Posicional (cov_pos_trace)

O traço da covariância posicional (`cov_xx + cov_yy`) representa a incerteza total na estimativa de posição. Valores menores indicam que o filtro está mais confiante na localização.

In [71]:
col = 'cov_pos_trace'

x_min = min(loc_2cam[col].min(), loc_1cam[col].min())
x_max = max(loc_2cam[col].max(), loc_1cam[col].max())

fig = make_subplots(rows=1, cols=2, column_titles=['Duas Câmeras', 'Uma Câmera'],
                    shared_xaxes=True, shared_yaxes=True)

fig.add_trace(go.Histogram(
    x=loc_2cam[col], marker_color='royalblue', opacity=0.7, showlegend=False,
    xbins=dict(start=x_min, end=x_max, size=(x_max - x_min) / 100)
), row=1, col=1)
fig.add_trace(go.Histogram(
    x=loc_1cam[col], marker_color='tomato', opacity=0.7, showlegend=False,
    xbins=dict(start=x_min, end=x_max, size=(x_max - x_min) / 100)
), row=1, col=2)

fig.update_xaxes(title_text=col, range=[x_min, x_max])
fig.update_yaxes(title_text='Frames', row=1, col=1)
fig.update_layout(
    title='Distribuição da Covariância Posicional (cov_pos_trace)',
    height=400, width=900, hovermode='closest'
)
apply_ieee_style(fig)
save_fig(fig, 'cov_pos_trace_histogram.svg')
fig.show()

##### Mediana da Covariância por Run

O histograma mostra a distribuição agregada de todos os frames, mas não revela se o comportamento é consistente entre execuções. A mediana da `cov_pos_trace` calculada por run permite verificar se há runs com incerteza sistematicamente maior — o que indicaria problemas pontuais de localização em certas execuções, e não apenas variação natural dentro de uma run.

Valores mais baixos indicam que o RTABMAP estava mais confiante na estimativa de posição ao longo daquela execução.

In [72]:
col = 'cov_pos_trace'

med_2cam_run = loc_2cam.groupby('run')[col].median().reset_index(name='mediana')
med_1cam_run = loc_1cam.groupby('run')[col].median().reset_index(name='mediana')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=med_2cam_run['run'], y=med_2cam_run['mediana'],
    name='Duas câmeras', marker_color='royalblue', opacity=0.8
))
fig.add_trace(go.Bar(
    x=med_1cam_run['run'], y=med_1cam_run['mediana'],
    name='Uma câmera', marker_color='tomato', opacity=0.8
))
fig.update_layout(
    title='Mediana da Covariância Posicional (cov_pos_trace) por Run',
    xaxis_title='Run', yaxis_title='Mediana cov_pos_trace',
    barmode='group', hovermode='closest', width=1000
)
apply_ieee_style(fig)
save_fig(fig, 'cov_pos_trace_by_run.svg')
fig.show()

print(f'Mediana global — duas câmeras : {med_2cam_run["mediana"].median():.6f}')
print(f'Mediana global — uma câmera   : {med_1cam_run["mediana"].median():.6f}')
melhor = 'Duas câmeras' if med_2cam_run['mediana'].median() < med_1cam_run['mediana'].median() else 'Uma câmera'
print(f'\n-> {melhor} apresentou menor incerteza mediana por run.')

Mediana global — duas câmeras : 0.009417
Mediana global — uma câmera   : 0.005354

-> Uma câmera apresentou menor incerteza mediana por run.


#####  3.2.4 Tempo de Detecção

Custo computacional do ciclo de localização. Duas câmeras processa mais dados visuais — espera-se tempo maior. O `detection_time_ms` é o tempo da etapa de reconhecimento visual; o `total_time_ms` inclui todas as etapas do RTABMAP.

In [73]:
fig = go.Figure()
fig.add_trace(go.Box(
    y=loc_2cam['detection_time_ms'], name='Duas câmeras',
    marker_color='royalblue', boxpoints='outliers'
))
fig.add_trace(go.Box(
    y=loc_1cam['detection_time_ms'], name='Uma câmera',
    marker_color='tomato', boxpoints='outliers'
))
fig.update_layout(
    title='Tempo de Detecção — Duas Câmeras vs Uma Câmera',
    yaxis_title='ms', hovermode='closest'
)
apply_ieee_style(fig)
save_fig(fig, 'detection_time_boxplot.svg')
fig.show()

print(f'Mediana detection_time_ms — duas câmeras : {loc_2cam["detection_time_ms"].median():.1f} ms')
print(f'Mediana detection_time_ms — uma câmera   : {loc_1cam["detection_time_ms"].median():.1f} ms')


Mediana detection_time_ms — duas câmeras : 386.4 ms
Mediana detection_time_ms — uma câmera   : 252.4 ms


##### Análise dos Picos de Tempo de Processamento

Além da mediana mais alta, o grupo de duas câmeras apresentou picos de `detection_time_ms` de até **806 ms**, mais ou menos 2× acima do pico de uma câmera (~411 ms). Nesses instantes, o sistema de navegação opera sem atualização de pose por múltiplos ciclos de controle do NAV2, dependendo exclusivamente da odometria do robô.

Em ambientes com obstáculos próximos ou corredores estreitos, uma falha de atualização de pose de 806 ms representa um **risco operacional**, o robô navega às cegas por **806 ms** consecutivos sem correção visual da pose.

In [74]:
# Identifica o pico máximo de total_time_ms em cada grupo
peak_2cam = loc_2cam['total_time_ms'].max()
peak_1cam = loc_1cam['total_time_ms'].max()
med_2cam  = loc_2cam['total_time_ms'].median()
med_1cam  = loc_1cam['total_time_ms'].median()

print('=' * 50)
print(f'  total_time_ms — duas câmeras')
print(f'    mediana : {med_2cam:.1f} ms')
print(f'    pico    : {peak_2cam:.1f} ms  ({peak_2cam/med_2cam:.1f}× a mediana)')
print('-' * 50)
print(f'  total_time_ms — uma câmera')
print(f'    mediana : {med_1cam:.1f} ms')
print(f'    pico    : {peak_1cam:.1f} ms  ({peak_1cam/med_1cam:.1f}× a mediana)')
print('=' * 50)


  total_time_ms — duas câmeras
    mediana : 413.6 ms
    pico    : 834.0 ms  (2.0× a mediana)
--------------------------------------------------
  total_time_ms — uma câmera
    mediana : 262.8 ms
    pico    : 418.9 ms  (1.6× a mediana)


### 4. Conclusão

A análise foi conduzida em duas frentes complementares: qualidade geométrica das trajetórias e métricas internas de localização do RTABMAP.

**Trajetórias — desempenho equivalente**  
O RMSE entre a mediana das trajetórias e o plano calculado foi de **0.1122 m** (duas câmeras) e **0.1104 m** (uma câmera), diferença de apenas 1.8 mm. Do ponto de vista do resultado final de navegação, os dois modos são indistinguíveis.

**Métricas internas — perfis distintos**

| Métrica | Duas câmeras | Uma câmera | Vencedor |
|---|---|---|---|
| Taxa de falha (`inlier_ratio = 0`) mediana | 31.4% | 23.5% | Uma câmera |
| `inlier_ratio` mediano | 0.0958 | 0.1480 | Uma câmera |
| Frames com `hypothesis_ratio = 1` | 70.9% | 45.6% | Duas câmeras |
| `cov_pos_trace` mediana | 0.0094 | 0.0053 | Uma câmera |
| `detection_time_ms` mediana | 386 ms | 252 ms | Uma câmera |

**Interpretação**  
Os dois modos chegam ao mesmo lugar pela mesma rota, mas por caminhos internos diferentes. Duas câmeras reconhecem lugares com mais confiança (maior `hypothesis_ratio`) graças ao campo de visão ampliado, mas pagam um custo maior de processamento e apresentam maior incerteza de pose. Uma câmera processa mais rápido, produz poses mais precisas e features mais limpas, porém com menor confiança no reconhecimento de lugar global.

**Fator crítico: Limitação de hardware**  
Um aspecto determinante para interpretar os resultados do modo de duas câmeras é a arquitetura do hardware utilizado. O BotBrainPro é equipado com uma Jetson AGX series que, embora possua dois barramentos USB 3.0, teve ambas as câmeras RealSense conectadas no mesmo barramento. Com duas câmeras ativas no mesmo barramento simultaneamente, o volume de dados transmitidos o satura, causando perda de frames. Essa contenção explica diretamente o `inlier_ratio` mais baixo e a maior taxa de falha de localização observados no grupo de duas câmeras. Em hardware com barramentos independentes por câmera, o resultado do modo de duas câmeras tenderia a ser significativamente melhor.

**Pontos positivos de duas câmeras**  
É importante não reduzir o valor de duas câmeras apenas à métrica de localização autônoma. O produto possui também um modo de **teleoperação**, no qual o operador humano pilota o robô remotamente. Nesse contexto, ter uma câmera frontal e uma traseira amplia significativamente o senso de posicionamento do operador, reduzindo pontos cegos e facilitando manobras. Para esse caso de uso, este ganho pode superar o resultado desse benchmark.

**Recomendação**  
Dado o gargalo de hardware identificado, a comparação não reflete o potencial real de duas câmeras em navegação autônoma. Para este benchmark e este hardware específico, uma câmera entrega resultado equivalente a um custo computacional menor. Uma nova avaliação com barramento dedicado por câmera seria necessária para uma conclusão definitiva sobre a superioridade do modo estéreo em autonomia.

De forma geral, antes de assumir que duas câmeras são sempre melhor que uma **para navegação autônoma**, **a primeira pergunta deve ser sobre o hardware**: o sistema tem capacidade de banda, barramentos e processamento suficientes para sustentar o fluxo de dados de múltiplas câmeras? Se não, adicionar uma segunda câmera pode prejudicar mais do que ajudar, gerando perda de frames, saturação de barramento e degradação da localização exatamente nos momentos em que ela mais importa. Para **teleoperação**, a equação é diferente: a segunda câmera entrega valor direto ao operador, e deve ser considerada independentemente das métricas de localização autônoma.